In [ ]:
# 仓库根目录加入 sys.path（本仓库自包含，不依赖外部绝对路径）
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import torch
from torch import nn
from d2l import torch as d2l
from deepseek_tokenizer import ds_token

from motex_utils.transformer import (AddNorm, PositionWiseFFN,
                                     transpose_output, transpose_qkv, DotProductAttention)


GPT
预训练
- L1 预测下一个词
微调
- L1 预测下一个词
- L2 预估句子的标号
L = L2 + γ L1

In [ ]:
def precompute_rotary_emb(max_seq_len, d, base=10000):
    theta = 1.0 / (base ** (torch.arange(0, d, 2, dtype=torch.float) / d))
    positions = torch.arange(max_seq_len, dtype=torch.float)
    angles = positions.unsqueeze(1) * theta.unsqueeze(0)  # (max_seq_len, d//2)
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    return cos, sin

def apply_rotary_pos_emb(x, cos, sin, offset=0):
    seq_len = x.shape[-2]
    d = x.shape[-1]
    half_d = d // 2

    # 确保 cos/sin 维度正确
    # [问题] 原版无 offset：增量解码时 query 永远按位置 0 编码，与训练时的绝对位置不一致。
    # [解决] 增加 offset 参数，取 cos[offset:offset+seq_len]，使解码时能按其真实绝对位置做 RoPE。
    cos = cos[offset:offset + seq_len, :].to(x.device)  # (seq_len, half_d)
    sin = sin[offset:offset + seq_len, :].to(x.device)
    while cos.dim() < x.dim():
        cos = cos.unsqueeze(0)
        sin = sin.unsqueeze(0)

    x_left = x[..., :half_d]
    x_right = x[..., half_d:]
    x_rotated_left = x_left * cos - x_right * sin
    x_rotated_right = x_left * sin + x_right * cos
    return torch.cat([x_rotated_left, x_rotated_right], dim=-1)

In [ ]:
class RopeMultiHeadAttention(nn.Module):
    def __init__(self, key_size, query_size, value_size, num_hiddens, num_heads, dropout, max_seq_len, bias=False, **kwargs):
        super(RopeMultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.head_dim = num_hiddens // num_heads  # 重要修正
        self.attention = DotProductAttention(dropout)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=bias)
        self.W_k = nn.Linear(key_size, num_hiddens, bias=bias)
        self.W_v = nn.Linear(value_size, num_hiddens, bias=bias)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)
        self.cos, self.sin = precompute_rotary_emb(max_seq_len,  self.head_dim)

    def forward(self, queries, keys, values, valid_lens):
        queries = transpose_qkv(self.W_q(queries), self.num_heads)        
        keys = transpose_qkv(self.W_k(keys), self.num_heads)        
        values = transpose_qkv(self.W_v(values), self.num_heads)       
         
        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(valid_lens, repeats=self.num_heads, dim=0)

        # [问题] 原版 query/keys 都以 offset=0 做 RoPE：训练时整段输入无妨，
        #       但增量解码时 keys 是『历史+当前』(0..k)，而 query 只含当前 token，
        #       若仍按位置 0 旋转，query 的位置编码与训练不一致，生成质量受损。
        # [解决] query 的绝对起始位置 = keys 长度 - 当前 query 长度；keys 覆盖 0..len-1 全程仍用 offset=0。
        query_offset = keys.shape[1] - queries.shape[1]
        queries = apply_rotary_pos_emb(queries, self.cos, self.sin, offset=query_offset)
        keys = apply_rotary_pos_emb(keys, self.cos, self.sin)
        output = self.attention(queries, keys ,values, valid_lens)
        
        output_concat = transpose_output(output, self.num_heads)
        return self.W_o(output_concat)


In [ ]:
class GPTDecoderBlock(nn.Module):
    def __init__(self, query_size, key_size,value_size,num_hiddens, norm_shape, ffn_num_input, ffn_num_hiddens,
                 num_heads, dropout, i, max_seq_len):
        super().__init__()
        self.i = i
        self.attention = RopeMultiHeadAttention(
            key_size=key_size,
            query_size=query_size,
            value_size=value_size,
            num_hiddens=num_hiddens,
            num_heads=num_heads,
            dropout=dropout,
            max_seq_len=max_seq_len
        )
        self.addnorm1 = AddNorm(norm_shape, dropout)
        self.ffn = PositionWiseFFN(ffn_num_input, ffn_num_hiddens, num_hiddens)
        self.addnorm2 = AddNorm(norm_shape, dropout)

    def forward(self, X, state=None, valid_lens=None):
        """
        X: (batch, seq_len, num_hiddens)
        state: (K, V) 历史缓存（仅推理时使用）
        valid_lens: (batch, seq_len) 因果掩码（训练时传入，推理时传 None）
        """
        if state[self.i] is None:
            key_values = X
        else:
            key_values = torch.cat((state[self.i], X), axis=1)
            
        if not self.training:
            state[self.i] = key_values
        attn_output = self.attention(X, key_values, key_values, valid_lens)
        
        Y = self.addnorm1(X, attn_output)
        Z = self.addnorm2(Y, self.ffn(Y))
        return Z, state  # 返回更新后的 state

In [ ]:
class GPTDecoder(nn.Module):
    def __init__(self, query_size, key_size, value_size, 
                 num_layers,num_hiddens, num_heads,norm_shape, 
                 vocabs_size, ffn_num_input, ffn_num_hiddens, dropout,max_seq_len):
        super().__init__()
        self.token_embedding = nn.Embedding(vocabs_size, num_hiddens)
        self.blks = nn.Sequential()
        self.num_layers = num_layers
        
        for i in range(num_layers):
            self.blks.add_module(
                f"{i}", GPTDecoderBlock(query_size=query_size, key_size=key_size, 
                                     value_size=value_size, num_hiddens=num_hiddens,
                                     norm_shape=norm_shape, ffn_num_input=ffn_num_input,
                                    ffn_num_hiddens=ffn_num_hiddens,num_heads=num_heads, dropout=dropout,max_seq_len=max_seq_len, i=i)
            )

    def forward(self, tokens, valid_lens, state=None):
        X = self.token_embedding(tokens)
        if state is None:
            state = [None] * self.num_layers
        for blk in self.blks:
            X, state = blk(X, state, valid_lens)
        return X,state
        

In [ ]:
class GPTModel(nn.Module):
    def __init__(self,query_size, key_size, value_size, 
                 num_layers,num_hiddens, num_heads,norm_shape, 
                 vocabs_size, ffn_num_input, ffn_num_hiddens, dropout,max_seq_len):
        super().__init__()
        self.dense = nn.Linear(num_hiddens, vocabs_size)
        self.decoder = GPTDecoder(query_size, key_size, value_size, 
                 num_layers,num_hiddens, num_heads,norm_shape, 
                 vocabs_size, ffn_num_input, ffn_num_hiddens, dropout,max_seq_len)
    
    def forward(self, X ,valid_lens, state=None):
        X, state = self.decoder(X, valid_lens,state)
        return self.dense(X), state

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)

In [ ]:
def evaluate_gpt(net, test_iter, device):
    """在测试集上计算准确率"""
    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for tokens, labels, valid_lens in test_iter:
            tokens = tokens.to(device)
            labels = labels.to(device)
            valid_lens = valid_lens.to(device)
            logits, _ = net(tokens, valid_lens, None)
            preds = logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.numel()
    net.train()
    return correct / total if total > 0 else 0

In [ ]:
import os
import glob
import torch
from d2l import torch as d2l
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()
def train_gpt_ckpt(net, loss, train_iter, test_iter, vocab_size, devices, num_steps,
              lr=1e-4, warmup_steps=100, weight_decay=0.01,
              ckpt_dir='./checkpoints', save_every=100, max_keep=50,
              resume_from=None):
    """
    训练 GPT 模型（因果语言模型）
    支持 checkpoint 保存 / 恢复，并绘制 train loss, train acc, test acc
    """
    os.makedirs(ckpt_dir, exist_ok=True)
    
    net.apply(init_weights) 
    net = net.to(devices)
    
    optimizer = torch.optim.AdamW(
        net.parameters(),
        lr=lr,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=weight_decay
    )
    
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        return max(0.0, float(num_steps - step) / float(max(1, num_steps - warmup_steps)))
    
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    
    # ---------- 恢复检查点 ----------
    start_step = 0
    best_loss = float('inf')
    best_model_path = os.path.join(ckpt_dir, 'best_model.pth')
    
    if resume_from is not None and os.path.isfile(resume_from):
        checkpoint = torch.load(resume_from, map_location=devices)
        net.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_step = checkpoint['step']
        best_loss = checkpoint.get('best_loss', float('inf'))
        total_loss = checkpoint.get('total_loss', 0.0)
        total_tokens = checkpoint.get('total_tokens', 0)
        print(f"从 {resume_from} 恢复，继续从 step {start_step} 训练")
    else:
        total_loss, total_tokens = 0.0, 0
        # print("从头开始训练")
    
    # ---------- 绘图 ----------
    animator = d2l.Animator(
        xlabel='step', ylabel='loss/acc',
        xlim=[1, num_steps],
        legend=['train loss', 'train acc', 'test acc']
    )
    
    timer, step = d2l.Timer(), start_step
    # 累积指标
    train_acc_sum, train_acc_count = 0.0, 0
    test_acc_cache = None  # 用于缓存最近一次测试准确率
    
    net.train()
    while step < num_steps:
        for batch in train_iter:
            tokens, labels, valid_lens = batch
            tokens = tokens.to(devices)
            labels = labels.to(devices)
            valid_lens = valid_lens.to(devices)
            
            optimizer.zero_grad()
            timer.start()
            with torch.amp.autocast('cuda'):
                logits, _ = net(tokens, valid_lens, None)
                l = loss(logits.reshape(-1, vocab_size), labels.reshape(-1))
            scaler.scale(l).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        
            scaler.step(optimizer)
            scaler.update()
            
            
            scheduler.step()
            timer.stop()
            
            # 计算当前 batch 的准确率（预测正确的 token 比例）
            preds = logits.argmax(dim=-1)  # (batch, seq_len)
            correct = (preds == labels).sum().item()
            total = labels.numel()
            batch_acc = correct / total
            
            # 累积训练指标（用于平均）
            batch_loss = l.item()
            batch_tokens = total  # labels 中全部是有效 token
            total_loss += batch_loss * batch_tokens
            total_tokens += batch_tokens
            train_acc_sum += correct
            train_acc_count += total
            
            step += 1
            
            # 每 save_every 步保存 checkpoint
            if step % save_every == 0:
                # 保存当前模型（含优化器、调度器状态）
                ckpt_name = f'model_step_{step:06d}.pth'
                ckpt_path = os.path.join(ckpt_dir, ckpt_name)
                torch.save({
                    'model_state_dict': net.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'step': step,
                    'best_loss': best_loss,
                    'total_loss': total_loss,
                    'total_tokens': total_tokens,
                }, ckpt_path)
                # print(f'Checkpoint saved at step {step}: {ckpt_path}')
                
                # 管理保存版本数量
                all_ckpts = sorted(glob.glob(os.path.join(ckpt_dir, 'model_step_*.pth')))
                if len(all_ckpts) > max_keep:
                    os.remove(all_ckpts[0])
                    # print(f'Removed old checkpoint: {all_ckpts[0]}')
            
            # 每 100 步评估一次测试集（如果 test_iter 不为 None）
            if test_iter is not None and step % 100 == 0:
                test_acc = evaluate_gpt(net, test_iter, devices)
                test_acc_cache = test_acc
                # print(f'Step {step}: Test Acc = {test_acc:.4f}')
            
            # 计算当前平均训练损失和准确率
            avg_loss = total_loss / total_tokens if total_tokens > 0 else 0
            avg_train_acc = train_acc_sum / train_acc_count if train_acc_count > 0 else 0
            
            # 更新最佳模型
            if avg_loss < best_loss:
                if step % save_every == 0:
                    best_loss = avg_loss
                    torch.save(net.state_dict(), best_model_path)
                # print(f'New best model saved at step {step} with loss {avg_loss:.4f}')
            
            # 更新动画（每 10 步或最后一步）
            if step % 10 == 0 or step == num_steps:
                animator.add(step, (avg_loss, avg_train_acc, test_acc_cache if test_acc_cache is not None else 0.0))
            
            if step >= num_steps:
                break
    
    # 最终统计
    avg_loss = total_loss / total_tokens if total_tokens > 0 else 0
    avg_train_acc = train_acc_sum / train_acc_count if train_acc_count > 0 else 0
    print(f'训练完成！平均损失 = {avg_loss:.4f}, 平均训练准确率 = {avg_train_acc:.4f}')
    print(f'训练速度: {total_tokens / timer.sum():.1f} tokens/sec on {str(devices)}')
    print(f'最佳模型保存在: {best_model_path}')

In [ ]:
def predict_gpt(net, prompt, max_new_tokens, device,
                bos_token_id=None, eos_token_id=None):
    net.eval()
    prompt_ids = ds_token.encode(prompt)
    # 1. 构造初始输入（包含 BOS 和 prompt）
    if bos_token_id is not None:
        input_ids = torch.tensor([[bos_token_id[0]] + prompt_ids], device=device)
    else:
        input_ids = torch.tensor([prompt_ids], device=device)

    state = None
    generated_ids = []

    with torch.no_grad():
        # 预填充：传入整个 prompt，得到第一个新 token 和初始化的 state
        logits, state = net(input_ids, valid_lens=None, state=state)
        next_token_id = logits[0, -1, :].argmax(dim=-1).item()

        # 自回归循环：每步只传当前新 token
        for _ in range(max_new_tokens):
            if next_token_id == eos_token_id[0]:
                break

            generated_ids.append(next_token_id)

            # 关键修正：input_ids 更新为当前 token（形状 1x1）
            input_ids = torch.tensor([[next_token_id]], device=device)
            logits, state = net(input_ids, valid_lens=None, state=state)
            next_token_id = logits[0, -1, :].argmax(dim=-1).item()

        output_ids = prompt_ids + generated_ids
        return ds_token.decode(output_ids, skip_special_tokens=True)

In [ ]:
devices = d2l.try_gpu()
loss = nn.CrossEntropyLoss()
batch_size, max_len, lr = 16, 256, 1e-4
num_steps = 20480

In [ ]:
net = GPTModel(vocabs_size = ds_token.vocab_size, num_hiddens=256, norm_shape=[256],
                    ffn_num_input=256, ffn_num_hiddens=256, num_heads=8,
                    num_layers=8, dropout=0.2, key_size=256, query_size=256,
                    value_size=256, max_seq_len=max_len)


In [ ]:
# ============================================================
# 数据加载（placeholder）
# 数据集与数据加载代码未随本仓库分发，请自行准备数据并在此接入，
# 例如：train_iter, test_iter = my_dataloader(batch_size, max_len)
# ============================================================
train_iter = test_iter = None  # TODO: 替换为你的数据接口后再运行
# train_gpt(net,train_iter, loss, ds_token.vocab_size, devices, num_steps, lr)
train_gpt_ckpt(net, loss, train_iter,train_iter,  ds_token.vocab_size, devices, num_steps, lr)

In [ ]:
output = predict_gpt(net, "你是谁", 128, devices, bos_token_id=ds_token.encode("<｜end▁of▁sentence｜>"), eos_token_id=ds_token.encode('<｜end▁of▁sentence｜>'))
print(output)